# Ancient Ambient Sound Level Comparison

This notebook compares two methods for estimating "ancient ambient" underwater noise levels:
- **Method 1 (ship-filtered)**: Excludes time windows with ship presence, using AIS-derived ship metrics
- **Method 2 (unfiltered)**: Uses rolling percentile statistics over longer windows without ship filtering

**Ship data analysis date range**: 2026-02-07 to 2026-02-13

> **Environment requirement**: This notebook requires the `orcasound` conda environment.
> Activate it before launching Jupyter: `conda activate orcasound`

In [ ]:
import sys
from datetime import datetime, timedelta
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, '../src')
from orcasound_noise.analysis.partitioned_accessor import PartitionedAccessor
from orcasound_noise.utils.hydrophone import Hydrophone

In [ ]:
# Analysis parameters
HYDROPHONE = Hydrophone.ORCASOUND_LAB
ANALYSIS_END = datetime(2026, 2, 13, 23, 59, 59)
SHIP_DATA_START = datetime(2026, 2, 7)
EXTENDED_START = datetime(2026, 1, 31)
AWS_PROFILE = "ambient-sound-team"
AWS_REGION = "us-west-2"
CONFIDENCE_THRESHOLD = 0.5
COMM_BAND = (500, 15000)
PERCENTILES = [0.05, 0.10, 0.25, 0.50]
METHOD1_WINDOWS = [1, 2, 5, 7]   # days, capped by ship data availability
METHOD2_WINDOWS = [1, 2, 3, 5, 7, 10, 14]  # days, no ship data required
S3_SHIP_METRICS = "s3://acoustic-sandbox/ambient-sound-analysis/temp_ship_metrics"

## 1. Ship Metrics Data Access

Load pre-computed ship track metrics from S3. Data covers 2026-02-07 to 2026-02-13 and is partitioned by year/month/day.

**Known data quirk**: The `bb_series` column contains a constant value of approximately −171.64 dB for many ship passages. This equals the negated `bb_ref` constant from `Hydrophone.ORCASOUND_LAB` and indicates that the broadband acoustic series was not populated for those passages (e.g., ship was too far from the hydrophone or there was an acoustic data gap during that time). This analysis uses raw acoustic data from `PartitionedAccessor` directly, not the pre-aggregated values in the ship metrics file.

In [ ]:
# Load a single day as a schema/data check
ship_sample = pl.scan_parquet(
    f"{S3_SHIP_METRICS}/year=2026/month=02/day=11/*.parquet",
    storage_options={"aws_profile": AWS_PROFILE, "aws_region": AWS_REGION}
).collect()

print(f"Shape: {ship_sample.shape}")
print(f"\nColumns ({len(ship_sample.columns)}):")
for col, dtype in zip(ship_sample.columns, ship_sample.dtypes):
    print(f"  {col}: {dtype}")
print(f"\nSample rows:")
ship_sample.head(3)

In [ ]:
# Verify expected columns are present
expected_cols = ['s_timestamp', 'l_timestamp', 'confidence', 'min_dist', 'is_isolated']
missing = [c for c in expected_cols if c not in ship_sample.columns]
assert not missing, f"Missing expected columns: {missing}"
assert ship_sample.shape[0] > 0, "Ship metrics sample returned 0 rows"
print(f"✓ Schema validation passed: {ship_sample.shape[0]} rows, {ship_sample.shape[1]} columns")
print(f"✓ All expected columns present: {expected_cols}")

## 2. Acoustic Data Access via PartitionedAccessor

Verify S3 access using a 5-minute smoke test before loading the full dataset. This also validates the bug fixes applied in this sprint to `get_time_range` and `get_broadband`.

In [ ]:
# Instantiate accessor (no time args in constructor)
accessor = PartitionedAccessor(HYDROPHONE)

# 5-minute broadband smoke test
t_start = datetime(2026, 2, 13, 0, 0, 0)
t_end = datetime(2026, 2, 13, 0, 5, 0)

bb_sample = accessor.get_time_range(t_start, t_end, psd=False)
print(f"Broadband sample shape: {bb_sample.shape}")
print(f"Broadband columns: {bb_sample.columns}")
print(f"Broadband '0' column dtype: {bb_sample['0'].dtype}")
print(bb_sample.head(3))

In [ ]:
# 5-minute comm band smoke test (validates the get_broadband bug fix)
comm_sample = accessor.get_broadband(t_start, t_end, COMM_BAND[0], COMM_BAND[1], ref=1)
print(f"Comm band sample shape: {comm_sample.shape}")
print(f"Comm band columns: {comm_sample.columns}")
print(comm_sample.head(3))

In [ ]:
# Validate both samples
assert bb_sample.shape[0] > 0, "Broadband returned 0 rows"
assert '0' in bb_sample.columns, "Broadband missing '0' column"
assert comm_sample.shape[0] > 0, "Comm band returned 0 rows"
assert 'sound_pressure_level_db' in comm_sample.columns, f"Comm band missing expected column; got: {comm_sample.columns}"

print("✓ Broadband access: OK")
print("✓ Comm band access (get_broadband fix validated): OK")
print(f"✓ Broadband range: {bb_sample['0'].min():.1f} to {bb_sample['0'].max():.1f} dB")
print(f"✓ Comm band range: {comm_sample['sound_pressure_level_db'].min():.1f} to {comm_sample['sound_pressure_level_db'].max():.1f} dB")

## Section 2: Ship Metrics Loading

In [ ]:
import os

# Ensure AWS profile is set for polars S3 access
os.environ["AWS_PROFILE"] = AWS_PROFILE

# Load all 7 days of ship metrics in a single scan_parquet call
ship_raw = pl.scan_parquet(
    f"{S3_SHIP_METRICS}/year=2026/month=2/**/*.parquet",
    storage_options={"aws_profile": AWS_PROFILE, "aws_region": AWS_REGION},
).collect()

# Parse s_timestamp and l_timestamp to Datetime if stored as strings or integers
for col in ("s_timestamp", "l_timestamp"):
    if ship_raw[col].dtype == pl.Utf8:
        ship_raw = ship_raw.with_columns(
            pl.col(col).str.to_datetime(time_unit="us").alias(col)
        )
    elif ship_raw[col].dtype in (pl.Int64, pl.Int32, pl.Float64):
        # Assume epoch seconds; cast to microseconds
        ship_raw = ship_raw.with_columns(
            (pl.col(col).cast(pl.Int64) * 1_000_000).cast(pl.Datetime("us")).alias(col)
        )

print(f"Total rows loaded: {ship_raw.shape[0]:,}")
print(f"Columns: {ship_raw.columns}")
print(f"s_timestamp dtype: {ship_raw['s_timestamp'].dtype}")
print(f"l_timestamp dtype: {ship_raw['l_timestamp'].dtype}")
print(f"Date range: {ship_raw['s_timestamp'].min()} -> {ship_raw['l_timestamp'].max()}")
ship_raw.head(3)

In [ ]:
# Apply confidence threshold filter
rows_before = ship_raw.shape[0]
ships_filtered = ship_raw.filter(pl.col("confidence") >= CONFIDENCE_THRESHOLD)
rows_after = ships_filtered.shape[0]

print(f"Rows before filter : {rows_before:,}")
print(f"Rows after filter  : {rows_after:,}  (confidence >= {CONFIDENCE_THRESHOLD})")
print(f"Rows removed       : {rows_before - rows_after:,}  ({(rows_before - rows_after) / rows_before * 100:.1f}%)")
print()
ships_filtered.head()

## Section 3: Ship Presence Mask

In [ ]:
# Build a per-second DataFrame spanning the full 7-day window
seconds_df = pl.DataFrame(
    {"ts": pl.datetime_range(SHIP_DATA_START, ANALYSIS_END, interval="1s", eager=True)}
)

# Mark each second as ship-present if it falls within any ship track interval
try:
    seconds_with_ships = (
        seconds_df
        .join_where(
            ships_filtered.select(["s_timestamp", "l_timestamp"]),
            pl.col("ts") >= pl.col("s_timestamp"),
            pl.col("ts") <= pl.col("l_timestamp"),
        )
        .select("ts")
        .unique()
    )
except AttributeError:
    # Fallback for polars < 0.19 (join_where not available)
    ship_present_series = pl.Series("ts", [], dtype=pl.Datetime("us"))
    for row in ships_filtered.select(["s_timestamp", "l_timestamp"]).iter_rows(named=True):
        mask = (seconds_df["ts"] >= row["s_timestamp"]) & (seconds_df["ts"] <= row["l_timestamp"])
        ship_present_series = pl.concat([ship_present_series, seconds_df.filter(mask)["ts"]])
    seconds_with_ships = pl.DataFrame({"ts": ship_present_series}).unique()

# Build presence mask
presence_mask = (
    seconds_df
    .with_columns(
        pl.col("ts").is_in(seconds_with_ships["ts"]).alias("ship_present")
    )
)

total_seconds = len(presence_mask)
ship_present_count = presence_mask["ship_present"].sum()
ship_free_count = total_seconds - ship_present_count

print(f"Total seconds      : {total_seconds:,}")
print(f"Ship-present count : {ship_present_count:,}  ({ship_present_count / total_seconds * 100:.1f}%)")
print(f"Ship-free count    : {ship_free_count:,}  ({ship_free_count / total_seconds * 100:.1f}%)")
presence_mask.head(5)

## Section 4: Ship-Free Fraction by Window

In [ ]:
# Compute ship-free fraction for each Method 1 window length
results = []
for d in METHOD1_WINDOWS:
    window_start = ANALYSIS_END - timedelta(days=d)
    window_slice = presence_mask.filter(
        (pl.col("ts") >= window_start) & (pl.col("ts") <= ANALYSIS_END)
    )
    total_seconds = len(window_slice)
    ship_free_seconds = (~window_slice["ship_present"]).sum()
    ship_free_pct = ship_free_seconds / total_seconds if total_seconds > 0 else None
    results.append({
        "window_days": d,
        "total_seconds": total_seconds,
        "ship_free_seconds": ship_free_seconds,
        "ship_free_pct": ship_free_pct,
    })

ship_free_df = pl.DataFrame(results)
ship_free_df